# Organisation and waterpoint names: high-cardinality audit

This focused notebook completes the initial **high-cardinality text/category** review of
`funder`, `installer`, `wpt_name`. It describes the supplied training and test predictors,
then uses the labelled training rows to identify relationships worth
carrying into a leakage-safe modelling pipeline.

These strings need coverage and memorisation checks before any encoding choice.

This is exploratory evidence, not fitted preprocessing. Category pooling,
imputation, encoding and scaling must be learned inside each training fold.


## Consistent audit contract

Every focused predictor audit answers the same questions before adding
type-specific checks:

1. What is explicitly missing, and what looks like a sentinel?
2. What range or category coverage is present in training and test?
3. How much of the test set is exposed to unseen training levels?
4. Does the labelled distribution vary enough to justify retaining the field?
5. What exact baseline treatment follows from the evidence?

Target-rate tables flag support rather than treating tiny groups as reliable.
Train/test comparisons are descriptive and do not use the hidden test labels.


In [1]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

stage_directory = next(
    candidate for candidate in [Path.cwd(), *Path.cwd().parents]
    if (candidate / "data" / "TrainingSetValues.csv").is_file()
)
source_directory = str((stage_directory / "src").resolve())
if source_directory not in sys.path:
    sys.path.insert(0, source_directory)

from predictor_audit import (
    MISSING_CATEGORY,
    analysis_categories,
    categorical_summary,
    categorical_target_profile,
    category_frequency_table,
    cramer_v,
    hierarchy_conflicts,
    hierarchy_summary,
    numeric_summary,
    numeric_target_summary,
    normalise_categories,
    sentinel_mask,
    source_blank_mask,
    text_normalisation_summary,
)
from source_data_validation import (
    validate_aligned_ids,
    validate_label_frame,
    validate_raw_feature_schema,
)

data_directory = stage_directory / "data"
training_features = pd.read_csv(
    data_directory / "TrainingSetValues.csv",
    keep_default_na=False,
)
training_labels = pd.read_csv(
    data_directory / "TrainingSetLabels.csv",
    keep_default_na=False,
)
test_features = pd.read_csv(
    data_directory / "TestSetValues.csv",
    keep_default_na=False,
)

validate_raw_feature_schema(training_features)
validate_raw_feature_schema(test_features)
validate_label_frame(training_labels)
validate_aligned_ids(training_features, training_labels)

training_data = training_features.merge(
    training_labels,
    on="id",
    validate="one_to_one",
)
audited_features = ['funder', 'installer', 'wpt_name']
assert set(audited_features).issubset(training_features.columns)

pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 80)
print(
    f"Validated {len(training_features):,} training rows and "
    f"{len(test_features):,} test rows for {len(audited_features)} predictors."
)


Validated 59,400 training rows and 14,850 test rows for 3 predictors.


## 1. Missingness, cardinality and test coverage

Source blanks, pandas nulls and configured sentinel strings are reported separately.
Semantic sentinels such as `unknown` stay visible in frequency and target tables;
they are not silently merged with blank values.
Rare means fewer than 50 training rows; it is a diagnostic threshold, not
a preprocessing choice. Total-variation distance compares marginal shares.


In [2]:
category_overview = categorical_summary(
    training_features,
    test_features,
    audited_features,
    rare_threshold=50,
    sentinel_tokens_by_column={'funder': ['0', 'none', 'unknown', 'not known'], 'installer': ['0', 'unknown', 'not known', '-', 'unknown installer'], 'wpt_name': ['none', 'unknown', 'not known']},
)
display(category_overview)


,training explicit missing,training source blank rows,training sentinel rows,test explicit missing,test source blank rows,test sentinel rows,training levels,test levels,training levels with <50 rows,training rows in rare levels (%),test-only levels,test rows in unseen levels (%),training-only levels,marginal total-variation distance
feature,,,,,,,,,,,,,,
funder,0,3635,810,0,869,210,1897,980,1753,14.89,243,1.71,1160,0.0823
installer,0,3655,826,0,877,210,1918,968,1790,14.45,224,1.57,1174,0.0755
wpt_name,0,0,3588,0,0,879,37398,10840,37362,82.53,8284,56.90,34842,0.6973


## 2. Most common values


In [3]:
for feature in audited_features:
    print()
    print(feature)
    display(category_frequency_table(training_features, test_features, feature, top_n=10))



funder


,training rows,training (%),test rows,test (%)
funder,,,,
government of tanzania,9084,15.29,2215,14.92
<missing/blank>,3635,6.12,869,5.85
danida,3114,5.24,793,5.34
hesawa,2202,3.71,580,3.91
rwssp,1374,2.31,329,2.22
world bank,1349,2.27,352,2.37
kkkt,1287,2.17,336,2.26
world vision,1246,2.1,316,2.13
unicef,1057,1.78,267,1.8



installer


,training rows,training (%),test rows,test (%)
installer,,,,
dwe,17405,29.3,4351,29.3
<missing/blank>,3655,6.15,877,5.91
government,1891,3.18,476,3.21
hesawa,1395,2.35,373,2.51
rwe,1206,2.03,292,1.97
commu,1065,1.79,289,1.95
danida,1050,1.77,256,1.72
district council,965,1.62,221,1.49
kkkt,910,1.53,225,1.52



wpt_name


,training rows,training (%),test rows,test (%)
wpt_name,,,,
none,3565,6.0,877,5.91
shuleni,1748,2.94,435,2.93
zahanati,830,1.4,204,1.37
msikitini,535,0.9,112,0.75
kanisani,323,0.54,67,0.45
bombani,271,0.46,52,0.35
sokoni,260,0.44,68,0.46
ofisini,254,0.43,67,0.45
school,208,0.35,52,0.35


## 3. Relationship with `status_group`

The tables display the most supported levels first and mark whether each
level has at least 100 training rows. Small groups are leads
for later validation, not stable target encodings.


In [4]:
for feature in audited_features:
    print()
    print(feature)
    profile = categorical_target_profile(
        training_data,
        feature,
        minimum_support=100,
    )
    display(profile.head(15))



funder


status_group,rows,meets support threshold,functional (%),functional needs repair (%),non functional (%)
funder,,,,,
government of tanzania,9084,True,40.95,7.72,51.33
<missing/blank>,3635,True,54.50,12.02,33.48
danida,3114,True,55.01,5.11,39.88
hesawa,2202,True,42.51,10.54,46.96
rwssp,1374,True,58.59,7.93,33.48
world bank,1349,True,40.40,7.19,52.41
kkkt,1287,True,56.18,5.13,38.69
world vision,1246,True,59.63,10.51,29.86
unicef,1057,True,56.76,9.37,33.87



installer


status_group,rows,meets support threshold,functional (%),functional needs repair (%),non functional (%)
installer,,,,,
dwe,17405,True,54.20,9.32,36.48
<missing/blank>,3655,True,54.72,12.04,33.24
government,1891,True,29.19,13.64,57.17
hesawa,1395,True,56.34,3.87,39.78
rwe,1206,True,25.21,11.36,63.43
commu,1065,True,68.17,3.00,28.83
danida,1050,True,51.62,7.90,40.48
district council,965,True,40.10,6.32,53.58
kkkt,910,True,46.70,6.81,46.48



wpt_name


status_group,rows,meets support threshold,functional (%),functional needs repair (%),non functional (%)
wpt_name,,,,,
none,3565,True,73.77,2.13,24.10
shuleni,1748,True,49.14,8.18,42.68
zahanati,830,True,51.81,9.40,38.80
msikitini,535,True,49.16,8.22,42.62
kanisani,323,True,47.99,6.50,45.51
bombani,271,True,58.30,7.75,33.95
sokoni,260,True,46.15,10.00,43.85
ofisini,254,True,43.31,4.33,52.36
school,208,True,41.83,6.25,51.92


## 4. Safe text normalisation and cross-field overlap

Only whitespace trimming and Unicode-aware case folding are counted here.
Fuzzy spelling merges would change identity and need an explicit reviewed map.


In [5]:
display(text_normalisation_summary(training_features, test_features, audited_features))

funder_missing = (
    source_blank_mask(training_features["funder"])
    | sentinel_mask(
        training_features["funder"],
        ["0", "none", "unknown", "not known"],
    )
)
installer_missing = (
    source_blank_mask(training_features["installer"])
    | sentinel_mask(
        training_features["installer"],
        ["0", "unknown", "not known", "-", "unknown installer"],
    )
)
normalised_funder = normalise_categories(training_features["funder"]).mask(funder_missing)
normalised_installer = normalise_categories(training_features["installer"]).mask(installer_missing)
both_observed = normalised_funder.notna() & normalised_installer.notna()
same_normalised_value = normalised_funder.eq(normalised_installer).fillna(False)
overlap = pd.DataFrame({
    "rows": [len(training_features)],
    "both observed": [both_observed.sum()],
    "same normalised value": [same_normalised_value.sum()],
    "same among observed (%)": [
        same_normalised_value.sum() / both_observed.sum() * 100
    ],
}, index=["training"]).round(2)
display(overlap)

pair_counts = pd.DataFrame({
    "funder": normalised_funder,
    "installer": normalised_installer,
}).dropna().value_counts().rename("rows").head(15)
display(pair_counts.to_frame())


raw levels  normalised levels  levels collapsed  \
feature   frame                                                       
funder    training        1897               1897                 0   
          test             980                980                 0   
installer training        2145               1918               227   
          test            1091                968               123   
wpt_name  training       37400              37398                 2   
          test           10840              10840                 0   

                    singleton levels  rows in singleton levels (%)  
feature   frame                                                     
funder    training               974                          1.64  
          test                   464                          3.12  
installer training               963                          1.62  
          test                   465                          3.13  
wpt_name  training             32926                         55.43  
          test                 10046                         67.65

,rows,both observed,same normalised value,same among observed (%)
training,59400,54861,18139,33.06


rows
funder                 installer         
government of tanzania dwe           4256
                       government    1637
hesawa                 dwe           1296
danida                 danida        1046
rwssp                  dwe            914
kkkt                   kkkt           906
hesawa                 hesawa         850
world vision           world vision   679
dhv                    dwe            678
government of tanzania rwe            645
danida                 dwe            626
                       danid          623
dwsp                   dwe            616
germany republi        ces            610
unicef                 dwe            562

## Training/test handoff


In [6]:
display(
    category_overview[[
        "training levels",
        "test levels",
        "test-only levels",
        "test rows in unseen levels (%)",
        "training rows in rare levels (%)",
        "marginal total-variation distance",
    ]].sort_values("test rows in unseen levels (%)", ascending=False)
)


,training levels,test levels,test-only levels,test rows in unseen levels (%),training rows in rare levels (%),marginal total-variation distance
feature,,,,,,
wpt_name,37398,10840,8284,56.90,82.53,0.6973
funder,1897,980,243,1.71,14.89,0.0823
installer,1918,968,224,1.57,14.45,0.0755


## Decision register

The register separates observed evidence from the proposed baseline action.
A retained field is still a candidate: later validation must show whether it
improves generalisation and whether a coarser related representation is safer.


In [7]:
decision_register = pd.DataFrame([{'feature': 'funder', 'quality finding': 'Effective missingness is 7.48%; 1.71% of test rows have unseen funders.', 'baseline treatment': 'Keep null/sentinel distinct, normalise conservatively, then rare-pool inside folds.', 'risk to verify': 'Naive target encoding leaks; direct one-hot encoding is unstable.'}, {'feature': 'installer', 'quality finding': 'Effective missingness is 7.54%; case normalisation reduces 2,145 to 1,918 levels.', 'baseline treatment': 'Keep separately from funder; trim/casefold/whitespace-normalise and rare-pool.', 'risk to verify': 'Aliases require cautious, reviewable normalisation rather than fuzzy merging.'}, {'feature': 'wpt_name', 'quality finding': '56.90% of test names are unseen; leave-one-out lookup accuracy is only 55.01%.', 'baseline treatment': 'Exclude raw exact names from baseline; test cross-fitted text/frequency features later.', 'risk to verify': 'Memorisation and geographic proxying can overstate local validation value.'}])
display(decision_register.set_index("feature"))


,quality finding,baseline treatment,risk to verify
feature,,,
funder,Effective missingness is 7.48%; 1.71% of test rows have unseen funders.,"Keep null/sentinel distinct, normalise conservatively, then rare-pool inside...",Naive target encoding leaks; direct one-hot encoding is unstable.
installer,"Effective missingness is 7.54%; case normalisation reduces 2,145 to 1,918 le...",Keep separately from funder; trim/casefold/whitespace-normalise and rare-pool.,"Aliases require cautious, reviewable normalisation rather than fuzzy merging."
wpt_name,56.90% of test names are unseen; leave-one-out lookup accuracy is only 55.01%.,Exclude raw exact names from baseline; test cross-fitted text/frequency feat...,Memorisation and geographic proxying can overstate local validation value.


### Handoff to modelling

Start with explicit unknown and rare buckets. Any frequency, hashing or target encoding must be fitted within folds and justified by validation.

- Preserve raw source frames and implement the stated sentinel rules on copies.
- Fit imputers, rare-level grouping and encoders on each training fold only.
- Map unseen validation or test categories to an explicit fallback.
- Compare the stated baseline treatment with a simple omission ablation.
- Revisit target-rate observations after the reproducible stratified split exists.
